# GH05T3 Fine-Tune — 2x T4
Standard transformers + PEFT (no unsloth — T4 CUDA arch incompatible)

**Kaggle settings:** Accelerator = GPU T4 x2 · Internet = On · Persistence = Files only

In [ ]:
# Cell 1: Install pinned deps
import subprocess, sys

# PyTorch 2.2+cu118: includes sm_60 (P100) and sm_75 (T4) kernels
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'torch==2.2.2+cu118', 'torchvision==0.17.2+cu118',
    '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)

# torchao on Kaggle is built for PyTorch 2.10 — remove it
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)

# Pin trl to 0.8.6: stable SFTTrainer API
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.40.2',
    'peft==0.10.0',
    'trl==0.8.6',
    'accelerate==0.29.3',
    'datasets==2.19.0',
    'huggingface_hub>=0.22.0'], check=True)

print('deps ready')

In [ ]:
# Cell 2: GPU check
import warnings
warnings.filterwarnings('ignore', category=FutureWarning, message='.*resume_download.*')
warnings.filterwarnings('ignore', category=UserWarning, message='.*use_reentrant.*')

import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')
assert torch.cuda.is_available(), 'No GPU — enable GPU in Kaggle settings'
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} | {p.total_memory/1e9:.1f} GB | sm_{p.major}{p.minor}')
t = torch.zeros(1, device='cuda'); t + 1
print('CUDA kernel OK')

In [ ]:
# Cell 3: Load data
import json, random, subprocess
from pathlib import Path
from datasets import Dataset

DATA_DIR = Path('/kaggle/input/gh05t3-datasets')
if not DATA_DIR.exists():
    print('Dataset not mounted — downloading via Kaggle CLI...')
    dl_dir = Path('/tmp/gh05t3-data')
    dl_dir.mkdir(exist_ok=True)
    subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', 'tatortot/gh05t3-datasets',
         '--path', str(dl_dir), '--unzip'],
        check=True
    )
    DATA_DIR = dl_dir

print(f'Data dir: {DATA_DIR}')
print('Files:', sorted(p.name for p in DATA_DIR.iterdir()))

SYSTEM = (
    'You are GH05T3, an autonomous security and reasoning agent. '
    'You think carefully, reason step-by-step, and always prioritize '
    'detection and defense over exploitation.'
)

def read_jsonl(p):
    if not p.exists():
        print(f'  missing: {p}')
        return
    with open(p) as f:
        for line in f:
            line = line.strip()
            if line:
                try: yield json.loads(line)
                except: pass

def chatml(msgs):
    return '\n'.join(f'<|im_start|>{m["role"]}\n{m["content"]}<|im_end|>' for m in msgs)

texts = []

for rec in read_jsonl(DATA_DIR / 'adversarial_defense.jsonl'):
    t = rec.get('threat_vector', '')
    if not t: continue
    texts.append(chatml([
        {'role':'system',    'content': SYSTEM},
        {'role':'user',      'content': f'Analyze this threat:\n\n{t}'},
        {'role':'assistant', 'content':
            f"**Exploitation:** {rec.get('exploitation_method','N/A')}\n\n"
            f"**Detection:** {rec.get('detection_pattern','N/A')}\n\n"
            f"**Mitigation:** {rec.get('mitigation_strategy','N/A')}"},
    ]))

for rec in read_jsonl(DATA_DIR / 'reasoning_chains.jsonl'):
    q = rec.get('question', '')
    s = rec.get('reasoning_steps', [])
    if not q or not isinstance(s, list): continue
    texts.append(chatml([
        {'role':'system',    'content': SYSTEM},
        {'role':'user',      'content': q},
        {'role':'assistant', 'content':
            '**Reasoning:**\n' + '\n'.join(f'{i+1}. {x}' for i,x in enumerate(s)) +
            f"\n\n**Answer:** {rec.get('final_answer','N/A')}"},
    ]))

for rec in read_jsonl(DATA_DIR / 'cve_patterns.jsonl'):
    p = rec.get('vulnerability_pattern', '')
    if not p: continue
    ind = rec.get('discovery_indicators', [])
    texts.append(chatml([
        {'role':'system',    'content': SYSTEM},
        {'role':'user',      'content': f"Analyze {rec.get('source_cve','CVE')} vulnerability."},
        {'role':'assistant', 'content':
            f"**Pattern:** {p}\n\n"
            f"**Indicators:**\n" + ('\n'.join(f'• {x}' for x in ind) if isinstance(ind,list) else str(ind)) +
            f"\n\n**Lessons:** {rec.get('defensive_lessons','N/A')}"},
    ]))

for rec in read_jsonl(DATA_DIR / 'bug_bounty.jsonl'):
    tgt = rec.get('target_system', '')
    vuln = rec.get('vulnerability_found', '')
    if not tgt or not vuln: continue
    texts.append(chatml([
        {'role':'system',    'content': SYSTEM},
        {'role':'user',      'content': f'Security research on {tgt}: {vuln}'},
        {'role':'assistant', 'content':
            f"**Recon:** {rec.get('recon_method','N/A')}\n\n"
            f"**PoC:** {rec.get('non_weaponized_poc','N/A')}\n\n"
            f"**Remediation:** {rec.get('remediation','N/A')}"},
    ]))

assert texts, f'No data found at {DATA_DIR}'
random.seed(42)
random.shuffle(texts)
dataset = Dataset.from_dict({'text': texts})
print(f'Dataset: {len(dataset)} examples')
print('Sample:\n', dataset[0]['text'][:300], '...')

In [ ]:
# Cell 4: Load model — 3B fp16
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen2.5-Coder-3B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='cuda:0',
    trust_remote_code=True,
)
model.config.use_cache = False

used  = torch.cuda.memory_allocated(0) / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'Model loaded — GPU 0: {used:.1f}/{total:.1f} GB used')

In [ ]:
# Cell 5: Apply LoRA — cast adapters to fp16 to match base model
import torch
from peft import LoraConfig, get_peft_model

LORA_RANK = 16

model.enable_input_require_grads()

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)

# Cast LoRA adapter weights to fp16 to match the base model dtype.
# PEFT initialises adapters in fp32 by default; running fp16 training with
# mixed-precision adapters causes the GradScaler to crash.  Casting here
# makes every trainable tensor fp16 so fp16=True works correctly.
for _, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float16)

model.print_trainable_parameters()
print('Adapter dtype:', next(p for p in model.parameters() if p.requires_grad).dtype)

In [ ]:
# Cell 6: Train
import torch
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir='outputs',
    max_steps=500,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    warmup_steps=50,
    learning_rate=2e-5,          # 10x lower than before — prevents gradient explosion
    fp16=True,                   # safe now that adapters are cast to fp16
    bf16=False,
    max_grad_norm=0.3,           # tighter clipping for stability
    logging_steps=10,
    optim='adamw_torch',
    lr_scheduler_type='cosine',
    seed=42,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=512,
    packing=False,
    args=training_args,
)

print('Training...')
stats = trainer.train()
print(f'Done — loss: {stats.training_loss:.4f}, steps: {stats.global_step}')

In [ ]:
# Cell 7: Save adapter locally
import json
OUT = 'gh05t3_lora_adapter'
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
with open(f'{OUT}/training_config.json', 'w') as f:
    json.dump({
        'model': MODEL_ID, 'lora_rank': LORA_RANK,
        'steps': stats.global_step, 'final_loss': stats.training_loss,
        'dataset_size': len(dataset),
    }, f, indent=2)
print(f'Adapter saved to /kaggle/working/{OUT}')

In [ ]:
# Cell 8: Push adapter to HuggingFace Hub
# Add HF_TOKEN to Kaggle Secrets (Add-ons → Secrets) for auto-push.
import os
from huggingface_hub import HfApi

HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN', '')
HF_REPO  = 'tastytator/gh05t3-lora-adapter'

if HF_TOKEN:
    print(f'Uploading adapter to {HF_REPO} ...')
    api = HfApi(token=HF_TOKEN)
    try:
        api.create_repo(HF_REPO, repo_type='model', exist_ok=True, private=True)
        api.upload_folder(
            folder_path=OUT,
            repo_id=HF_REPO,
            repo_type='model',
            commit_message=f'GH05T3 LoRA v2 — {stats.global_step} steps, loss {stats.training_loss:.4f}',
        )
        print(f'Uploaded to https://huggingface.co/{HF_REPO}')
    except Exception as e:
        print(f'HF upload failed (non-fatal): {e}')
        print(f'Adapter still at /kaggle/working/{OUT}')
else:
    print('HF_TOKEN not set — adapter is at /kaggle/working/', OUT)

In [ ]:
# Cell 9: Inference test — verifies the adapter produces coherent output
model.eval()
prompt = (
    '<|im_start|>system\nYou are GH05T3.<|im_end|>\n'
    '<|im_start|>user\nExplain SQL injection detection.<|im_end|>\n'
    '<|im_start|>assistant\n'
)
inputs = tokenizer(prompt, return_tensors='pt').to('cuda:0')
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False,
                         repetition_penalty=1.1)
response = tokenizer.decode(out[0], skip_special_tokens=True)
print(response)

# Sanity check — flag if output is degenerate
assistant_part = response.split('assistant')[-1].strip()
unique_chars = len(set(assistant_part[:100]))
if unique_chars < 5:
    print(f'WARNING: degenerate output ({unique_chars} unique chars) — check training loss curve')
else:
    print(f'Output OK ({unique_chars} unique chars in first 100)')